## load the data into a SQL database

In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)
c = "cleaned/"
con = sqlite3.connect(":memory:")
for name in ["orders", "order_items", "customers", "payments", "returns", "inventory", "products"]:
    pd.read_csv(c + name + ".csv").to_sql(name, con, index=False)

def q(sql):
    return pd.read_sql_query(sql, con)

## The KPI queries

In [2]:
print(q("SELECT ROUND(SUM(final_amount)/1e7,2) AS gmv_crore FROM orders"))
print(q("SELECT ROUND(SUM(final_amount)/1e7,2) AS net_revenue_crore, COUNT(*) AS delivered_orders, ROUND(AVG(final_amount)) AS aov FROM orders WHERE status='Delivered'"))
print(q("SELECT ROUND(100.0*SUM(status='Cancelled')/COUNT(*),1) AS cancellation_rate_pct FROM orders"))
print(q("SELECT ROUND(100.0*(SELECT COUNT(*) FROM returns)/(SELECT COUNT(*) FROM orders WHERE status='Delivered'),1) AS return_rate_pct"))
print(q("SELECT ROUND(100.0*SUM(status='Failed')/COUNT(*),2) AS payment_failure_pct FROM payments"))
print(q("SELECT ROUND(100.0*SUM(status='In Stock')/COUNT(*),1) AS fill_rate_pct FROM inventory"))

   gmv_crore
0     314.74
   net_revenue_crore  delivered_orders       aov
0             205.39             32499 63,198.00
   cancellation_rate_pct
0                  12.00
   return_rate_pct
0            30.80
   payment_failure_pct
0                 3.50
   fill_rate_pct
0          92.20


## category share, monthly trend, customer value, returns by order status

In [3]:
print(q("""
SELECT oi.category, ROUND(SUM(oi.total_price)/1e7,2) AS revenue_crore,
       ROUND(100.0*SUM(oi.total_price)/(SELECT SUM(total_price) FROM order_items oi2
             JOIN orders o2 ON oi2.order_id=o2.order_id WHERE o2.status='Delivered'),1) AS share_pct
FROM order_items oi JOIN orders o ON oi.order_id=o.order_id
WHERE o.status='Delivered' GROUP BY oi.category ORDER BY revenue_crore DESC LIMIT 3"""))

print(q("""
SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS orders,
       ROUND(SUM(final_amount)/1e7,2) AS gmv_crore
FROM orders GROUP BY 1 ORDER BY 1 LIMIT 4"""))

print(q("""
SELECT segment, COUNT(*) AS customers, ROUND(AVG(total_spent)) AS avg_lifetime_value
FROM customers GROUP BY segment ORDER BY avg_lifetime_value DESC"""))

print(q("""
SELECT o.status AS order_status, COUNT(*) AS returns
FROM returns r JOIN orders o ON r.order_id=o.order_id
GROUP BY o.status ORDER BY returns DESC"""))

           category  revenue_crore  share_pct
0       Electronics         107.64      57.70
1  Sports & Fitness          31.92      17.10
2    Home & Kitchen          15.76       8.50
     month  orders  gmv_crore
0  2023-06     444       2.74
1  2023-07    2081      12.89
2  2023-08    2091      12.81
3  2023-09    2104      13.75
    segment  customers  avg_lifetime_value
0   Premium       1501          532,735.00
1   Regular       4026          361,871.00
2       New       1488          274,866.00
3    Budget       2505          186,179.00
4  Inactive        480           32,249.00
  order_status  returns
0    Delivered     4327
1     Returned     3933
2    Cancelled     1067
3      Shipped      673
